<a href="https://colab.research.google.com/github/bautsuryakanta-dotcom/Text-Summarizer-program/blob/main/text_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers==4.44.0 -q
!pip install transformers torch torchvision torchaudio ipywidgets
!pip install --upgrade pip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [3]:
import transformers
print(transformers.__version__)
import warnings
import textwrap
import time
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import torch
warnings.filterwarnings('ignore')

# Configure display settings
WRAP_WIDTH = 80  # Width for text wrapping
SEPARATOR = "=" * 80

# Check if GPU is available for faster processing
device = 0 if torch.cuda.is_available() else -1
if torch.cuda.is_available():
    print("GPU is available! Using GPU for faster processing.")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. Using CPU (this may be slower).")

print("\n" + SEPARATOR)
print("INTERACTIVE TEXT SUMMARIZER - INITIALIZED")
print(SEPARATOR)


4.44.0
GPU is available! Using GPU for faster processing.
GPU Name: Tesla T4

INTERACTIVE TEXT SUMMARIZER - INITIALIZED


In [4]:
class HuggingFaceSummarizer:
    """
    A comprehensive text summarization class using Hugging Face pre-trained models.
    This class provides multiple summarization models and interactive features.
    """

    def __init__(self):
        """Initialize the summarizer with available models"""

        # Dictionary of available pre-trained models
        self.available_models = {
            "1": {
                "name": "facebook/bart-large-cnn",
                "description": "BART model fine-tuned on CNN news - Best for general news articles",
                "type": "Abstractive"
            },
            "2": {
                "name": "t5-small",
                "description": "T5 small model - Faster but less accurate, good for testing",
                "type": "Abstractive"
            },
            "3": {
                "name": "t5-base",
                "description": "T5 base model - Good balance of speed and quality",
                "type": "Abstractive"
            },
            "4": {
                "name": "google/pegasus-xsum",
                "description": "PEGASUS fine-tuned on extreme summarization - Very concise summaries",
                "type": "Abstractive"
            },
            "5": {
                "name": "facebook/bart-large-xsum",
                "description": "BART fine-tuned on extreme summarization - Creates very short summaries",
                "type": "Abstractive"
            }
        }

        self.current_model = None
        self.summarizer = None
        self.model_name = None
        self.tokenizer = None

        print("Summarizer class initialized!")
        print(f"Available models: {len(self.available_models)}")

    def display_available_models(self):
        """Display all available models with descriptions"""
        print("\n" + SEPARATOR)
        print("AVAILABLE PRE-TRAINED MODELS")
        print(SEPARATOR)

        for key, model_info in self.available_models.items():
            print(f"\n{key}. {model_info['name']}")
            print(f"Type: {model_info['type']}")
            print(f"Description: {model_info['description']}")

        print("\n" + SEPARATOR)

    def load_model(self, model_choice="1"):
        """
        Load a specific pre-trained model

        Args:
            model_choice (str): Key for the model to load
        """
        if model_choice not in self.available_models:
            print(f"Invalid model choice. Using default model.")
            model_choice = "1"

        self.model_name = self.available_models[model_choice]["name"]

        print(f"\n Loading model: {self.model_name}")
        print("This may take a few minutes on first run (downloading weights)...")

        start_time = time.time()

        try:
            # Load the tokenizer and model separately for more control
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)

            # Create the pipeline
            self.summarizer = pipeline(
                "summarization",
                model=self.model,
                tokenizer=self.tokenizer,
                device=device,
                framework="pt"
            )

            load_time = time.time() - start_time
            print(f" Model loaded successfully in {load_time:.2f} seconds!")
            print(f" Model parameters: {self.model.num_parameters():,}")

        except Exception as e:
            print(f"Error loading model: {str(e)}")
            print("Falling back to default pipeline...")

            # Fallback to simple pipeline
            self.summarizer = pipeline("summarization", device=device)
            print(" Default model loaded as fallback.")

    def analyze_text(self, text):
        """
        Analyze the input text and provide statistics

        Args:
            text (str): Input text to analyze

        Returns:
            dict: Text statistics
        """
        # Split into sentences (simple approach)
        sentences = text.replace('!', '.').replace('?', '.').split('.')
        sentences = [s.strip() for s in sentences if s.strip()]

        # Count words
        words = text.split()

        # Count characters
        chars = len(text)

        # Estimate reading time (average reading speed: 200 words per minute)
        reading_time_minutes = len(words) / 200
        reading_time_seconds = reading_time_minutes * 60

        stats = {
            "sentences": len(sentences),
            "words": len(words),
            "characters": chars,
            "reading_time": f"{int(reading_time_minutes)} min {int(reading_time_seconds % 60)} sec"
        }

        return stats

    def summarize_text(self, text, max_length=150, min_length=50, style="standard"):
        """
        Generate summary for the input text

        Args:
            text (str): Text to summarize
            max_length (int): Maximum length of summary
            min_length (int): Minimum length of summary
            style (str): Summary style - "standard", "concise", "detailed", or "bullet"

        Returns:
            dict: Summary and metadata
        """
        if self.summarizer is None:
            print(" No model loaded. Loading default model...")
            self.load_model()

        # Adjust parameters based on style
        if style == "concise":
            max_length = min(max_length, 50)
            min_length = min(min_length, 20)
        elif style == "detailed":
            max_length = max(max_length, 200)
            min_length = max(min_length, 80)
        elif style == "bullet":
            # Add prompt for bullet points
            text = "Summarize in bullet points: " + text

        print(f"\n Generating {style} summary...")
        start_time = time.time()

        try:
            # For very long texts, we might need to truncate
            if len(text.split()) > 1024:
                print(" Text is very long. Truncating to first 1024 words...")
                text = " ".join(text.split()[:1024])

            # Generate summary
            summary_result = self.summarizer(
                text,
                max_length=max_length,
                min_length=min_length,
                do_sample=False,  # Use greedy decoding for consistency
                num_beams=4,       # Beam search for better quality
                early_stopping=True,
                no_repeat_ngram_size=3  # Avoid repeating phrases
            )

            generation_time = time.time() - start_time

            # Get the summary text
            summary_text = summary_result[0]['summary_text']

            # Calculate compression metrics
            original_words = len(text.split())
            summary_words = len(summary_text.split())
            compression_ratio = (summary_words / original_words) * 100

            result = {
                "summary": summary_text,
                "original_words": original_words,
                "summary_words": summary_words,
                "compression_ratio": f"{compression_ratio:.1f}%",
                "generation_time": f"{generation_time:.2f} seconds",
                "style": style
            }

            return result

        except Exception as e:
            print(f" Error generating summary: {str(e)}")
            return None

    def display_summary(self, original_text, summary_result):
        """
        Display the original text and summary in a formatted way

        Args:
            original_text (str): Original input text
            summary_result (dict): Summary result from summarize_text method
        """
        if summary_result is None:
            print(" No summary to display.")
            return

        # Clear previous output for better display
        clear_output(wait=True)

        # Create HTML for better formatting
        display(HTML("""
        <style>
            .summary-container {
                font-family: Arial, sans-serif;
                max-width: 800px;
                margin: 20px auto;
                padding: 20px;
                border-radius: 10px;
                box-shadow: 0 0 10px rgba(0,0,0,0.1);
            }
            .original {
                background-color: #f0f0f0;
                padding: 15px;
                border-radius: 5px;
                margin-bottom: 20px;
            }
            .summary {
                background-color: #e3f2fd;
                padding: 15px;
                border-radius: 5px;
                margin-bottom: 20px;
            }
            .stats {
                background-color: #fff3e0;
                padding: 10px;
                border-radius: 5px;
                font-size: 14px;
            }
            .highlight {
                color: #1976d2;
                font-weight: bold;
            }
        </style>
        """))

        print(SEPARATOR)
        print("TEXT SUMMARIZATION RESULTS")
        print(SEPARATOR)

        # Original Text
        print("\n ORIGINAL TEXT:")
        print("-" * 40)
        print(textwrap.fill(original_text, width=WRAP_WIDTH))

        # Summary
        print("\n GENERATED SUMMARY:")
        print("-" * 40)
        print(textwrap.fill(summary_result["summary"], width=WRAP_WIDTH))

        # Statistics
        print("\n STATISTICS:")
        print("-" * 40)
        print(f"Original word count: {summary_result['original_words']} words")
        print(f"Summary word count: {summary_result['summary_words']} words")
        print(f"Compression ratio: {summary_result['compression_ratio']}")
        print(f"Generation time: {summary_result['generation_time']}")
        print(f"Summary style: {summary_result['style'].title()}")

        # Additional analysis
        original_stats = self.analyze_text(original_text)
        print(f"Original sentences: {original_stats['sentences']}")
        print(f"Estimated reading time: {original_stats['reading_time']}")

        print("\n" + SEPARATOR)


In [5]:

def create_interactive_interface():
    """
    Create an interactive interface using ipywidgets
    """

    # Initialize the summarizer
    summarizer = HuggingFaceSummarizer()

    # Create widgets
    model_dropdown = widgets.Dropdown(
        options=[(f"{v['name']} - {v['type']}", k) for k, v in summarizer.available_models.items()],
        value='1',
        description='Model:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='600px')
    )

    style_dropdown = widgets.Dropdown(
        options=['standard', 'concise', 'detailed', 'bullet'],
        value='standard',
        description='Style:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='200px')
    )

    max_length_slider = widgets.IntSlider(
        value=150,
        min=30,
        max=300,
        step=10,
        description='Max Length:',
        style={'description_width': 'initial'}
    )

    min_length_slider = widgets.IntSlider(
        value=50,
        min=20,
        max=150,
        step=10,
        description='Min Length:',
        style={'description_width': 'initial'}
    )

    text_input = widgets.Textarea(
        value='',
        placeholder='Paste your long paragraph here...',
        description='Input Text:',
        disabled=False,
        layout=widgets.Layout(width='100%', height='200px')
    )

    summarize_button = widgets.Button(
        description='Generate Summary',
        button_style='primary',
        icon='check'
    )

    clear_button = widgets.Button(
        description='Clear',
        button_style='warning',
        icon='times'
    )

    output_area = widgets.Output()

    # Sample texts for quick testing
    sample_texts = {
        "AI": """Artificial Intelligence (AI) is revolutionizing multiple industries by enabling machines to perform tasks that traditionally required human intelligence. In healthcare, AI algorithms analyze medical images with accuracy rivaling human experts, helping detect diseases earlier. Self-driving cars use AI to navigate roads safely, potentially reducing accidents. Educational platforms leverage AI for personalized learning experiences. However, AI also raises ethical concerns about privacy, job displacement, and algorithmic bias that need careful consideration.""",

        "Climate": """Climate change represents one of the most significant challenges facing our planet. Rising global temperatures have led to more frequent extreme weather events, melting polar ice caps, and rising sea levels. The primary cause is the increase in greenhouse gas emissions from human activities like burning fossil fuels and deforestation. Scientists warn that without immediate action to reduce emissions, we could face catastrophic consequences. However, renewable energy technologies like solar and wind power offer hope for a sustainable future.""",

        "Space": """Space exploration has entered a new era with both government agencies and private companies pushing boundaries. NASA's Artemis program aims to return humans to the Moon, while SpaceX develops Starship for Mars missions. The James Webb Space Telescope provides unprecedented views of the early universe. These endeavors advance scientific knowledge and drive technological innovation. However, challenges remain including space debris, radiation exposure, and the high costs of space travel."""
    }

    # Button click handlers
    def on_summarize_clicked(b):
        with output_area:
            output_area.clear_output()

            if not text_input.value.strip():
                print(" Please enter some text to summarize!")
                return

            # Load selected model if different
            if model_dropdown.value != summarizer.model_name:
                summarizer.load_model(model_dropdown.value)
            elif summarizer.summarizer is None:
                summarizer.load_model(model_dropdown.value)

            # Generate summary
            result = summarizer.summarize_text(
                text_input.value,
                max_length=max_length_slider.value,
                min_length=min_length_slider.value,
                style=style_dropdown.value
            )

            # Display results
            summarizer.display_summary(text_input.value, result)

    def on_clear_clicked(b):
        text_input.value = ''
        with output_area:
            output_area.clear_output()

    def on_sample_selected(change):
        if change['new'] != 'Select':
            text_input.value = sample_texts[change['new']]

    # Sample text dropdown
    sample_dropdown = widgets.Dropdown(
        options=['Select'] + list(sample_texts.keys()),
        value='Select',
        description='Sample:',
        style={'description_width': 'initial'}
    )
    sample_dropdown.observe(on_sample_selected, names='value')

    # Connect buttons
    summarize_button.on_click(on_summarize_clicked)
    clear_button.on_click(on_clear_clicked)

    # Arrange widgets
    model_box = widgets.VBox([
        widgets.HTML("<b>Step 1: Choose a Model</b>"),
        model_dropdown
    ])

    style_box = widgets.VBox([
        widgets.HTML("<b>Step 2: Adjust Parameters</b>"),
        widgets.HBox([style_dropdown, max_length_slider, min_length_slider])
    ])

    input_box = widgets.VBox([
        widgets.HTML("<b>Step 3: Enter Text</b>"),
        widgets.HBox([sample_dropdown]),
        text_input
    ])

    button_box = widgets.HBox([summarize_button, clear_button])

    # Display everything
    display(HTML("<h1> Hugging Face Interactive Text Summarizer</h1>"))
    display(HTML("<p>This tool uses state-of-the-art NLP models to summarize long texts.</p>"))
    display(model_box)
    display(style_box)
    display(input_box)
    display(button_box)
    display(output_area)

    return summarizer

In [6]:
# Create and display the interactive interface
summarizer_app = create_interactive_interface()

print("\n Interactive interface is ready!")
print("👉 Use the widgets above to:")
print("   1. Select a pre-trained model")
print("   2. Adjust summary parameters")
print("   3. Paste your text or choose a sample")
print("   4. Click 'Generate Summary' to see results")

Summarizer class initialized!
Available models: 5


Output()


 Interactive interface is ready!
👉 Use the widgets above to:
   1. Select a pre-trained model
   2. Adjust summary parameters
   3. Paste your text or choose a sample
   4. Click 'Generate Summary' to see results
